In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

original_df = pd.read_csv(r"C:\Users\rajab\202618066_Abhikarsh_Raj_DS605\202618066_challange_1\Data\train (1).csv")
test_df     = pd.read_csv(r"C:\Users\rajab\202618066_Abhikarsh_Raj_DS605\202618066_challange_1\Data\test.csv")

print("Train shape:", original_df.shape)
print("Test shape:", test_df.shape)

Train shape: (9864, 19)
Test shape: (2466, 18)


In [3]:
print(original_df.dtypes)
print("\n--- Missing values per column (train) ---")
print(original_df.isnull().sum())
print("\n--- First 5 rows ---")
original_df.head()

Session_ID                   int64
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                     float64
TrafficType                float64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object

--- Missing values per column (train) ---
Session_ID                   0
Administrative               0
Administrative_Duration    492
Informational                0
Informational_Duration       0
ProductRelated               0
ProductRelated_Duration      0
BounceRates                  0
ExitRates                  691

,Session_ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,112163,0,0.00,0,0.0,3,44.500000,0.066667,0.133333,0.000000,0.0,Dec,2,2,9.0,3.0,Returning_Visitor,False,False
1,107490,0,0.00,0,0.0,12,460.200000,0.061111,0.111111,0.000000,0.0,Jul,2,2,6.0,4.0,Returning_Visitor,False,False
2,106273,4,48.80,0,0.0,11,344.800000,0.015385,0.054396,0.000000,0.0,Oct,3,2,1.0,4.0,Returning_Visitor,True,False
3,110651,0,0.00,0,0.0,23,517.035714,0.000000,0.009524,23.300007,0.0,Dec,4,2,8.0,2.0,New_Visitor,False,True
4,101259,7,110.25,0,0.0,20,266.583333,0.011111,0.039753,0.000000,0.0,Mar,2,2,1.0,2.0,Returning_Visitor,False,False


In [4]:
TARGET = "Revenue"

df = original_df.copy()

id_col = "Session_ID" if "Session_ID" in df.columns else None
if id_col:
    df = df.drop(columns=[id_col])

y = df[TARGET]
X = df.drop(columns=[TARGET])

test_ids = test_df["Session_ID"]
X_test_raw = test_df.drop(columns=["Session_ID"]) if "Session_ID" in test_df.columns else test_df.copy()

print("X shape:", X.shape)
print("y value counts:\n", y.value_counts())

X shape: (9864, 17)
y value counts:
 Revenue
False    8211
True     1653
Name: count, dtype: int64


In [15]:
# Columns stored as integers but actually representing category codes, not true numeric scales
id_like_cols = ["OperatingSystems", "Browser", "Region", "TrafficType"]

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["int64", "float64"]).columns.tolist()

for col in id_like_cols:
    if col in numerical_cols:
        numerical_cols.remove(col)
        categorical_cols.append(col)

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

# ---------------------------------------------------
# Inspect actual data rows before encoding
# ---------------------------------------------------
print("\n--- Sample rows: categorical columns ---")
print(X[categorical_cols].head(10))

print("\n--- Sample rows: numerical columns ---")
print(X[numerical_cols].head(10))

print("\n--- Unique values per categorical column ---")
for col in categorical_cols:
    print(f"\n{col} unique values ({X[col].nunique()}):")
    print(X[col].unique())

Numerical columns: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']
Categorical columns: ['Month', 'VisitorType', 'Weekend', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

--- Sample rows: categorical columns ---
  Month        VisitorType Weekend  OperatingSystems  Browser  Region  \
0   Dec  Returning_Visitor   False               2.0      2.0     9.0   
1   Jul  Returning_Visitor   False               2.0      2.0     6.0   
2   Oct  Returning_Visitor    True               3.0      2.0     1.0   
3   Dec        New_Visitor   False               4.0      2.0     8.0   
4   Mar  Returning_Visitor   False               2.0      2.0     1.0   
5   Nov  Returning_Visitor   False               3.0      2.0     3.0   
6   Aug        New_Visitor   False               3.0      2.0     1.0   
7   Oct  Returning_Visitor   False               2.0

In [16]:
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X[numerical_cols] = num_imputer.fit_transform(X[numerical_cols])
if categorical_cols:
    X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])

X_test_raw[numerical_cols] = num_imputer.transform(X_test_raw[numerical_cols])
if categorical_cols:
    X_test_raw[categorical_cols] = cat_imputer.transform(X_test_raw[categorical_cols])

print("Remaining missing values in X:", X.isnull().sum().sum())
print("Remaining missing values in X_test_raw:", X_test_raw.isnull().sum().sum())

Remaining missing values in X: 0
Remaining missing values in X_test_raw: 0


In [17]:
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, columns=categorical_cols, drop_first=True)

X_encoded, X_test_encoded = X_encoded.align(X_test_encoded, join="left", axis=1, fill_value=0)

print("X_encoded shape:", X_encoded.shape)
print("X_test_encoded shape:", X_test_encoded.shape)

X_encoded shape: (9864, 68)
X_test_encoded shape: (2466, 68)


In [18]:
scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])
X_test_encoded[numerical_cols] = scaler.transform(X_test_encoded[numerical_cols])

X_encoded[numerical_cols].describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay
count,9.864000e+03,9.864000e+03,9.864000e+03,9.864000e+03,9.864000e+03,9.864000e+03,9.864000e+03,9.864000e+03,9.864000e+03,9.864000e+03
mean,-1.944916e-17,-6.627122e-17,-7.563563e-18,-5.906783e-17,-2.305086e-17,4.538138e-17,2.737290e-17,2.593222e-17,2.737290e-17,-1.440679e-18
std,1.000051e+00,1.000051e+00,1.000051e+00,1.000051e+00,1.000051e+00,1.000051e+00,1.000051e+00,1.000051e+00,1.000051e+00,1.000051e+00
min,-6.922950e-01,-4.382860e-01,-3.941177e-01,-2.451145e-01,-7.106947e-01,-6.125956e-01,-4.590890e-01,-8.920734e-01,-3.147125e-01,-3.083121e-01
25%,-6.922950e-01,-4.382860e-01,-3.941177e-01,-2.451145e-01,-5.537244e-01,-5.184197e-01,-4.590890e-01,-5.783842e-01,-3.147125e-01,-3.083121e-01
50%,-3.934940e-01,-3.988970e-01,-3.941177e-01,-2.451145e-01,-3.070568e-01,-3.070509e-01,-3.952768e-01,-3.586787e-01,-3.147125e-01,-3.083121e-01
75%,5.029090e-01,5.266957e-02,-3.941177e-01,-2.451145e-01,1.414296e-01,1.331707e-01,-1.096090e-01,1.018297e-01,-3.147125e-01,-3.083121e-01
max,7.375332e+00,1.868647e+01,1.852353e+01,1.817412e+01,1.509845e+01,3.230962e+01,3.685155e+00,3.282320e+00,1.894392e+01,4.716788e+00


In [19]:
processed_df = X_encoded.copy()
processed_df[TARGET] = y.values

print("processed_df shape:", processed_df.shape)
print("Missing values in processed_df:", processed_df.isnull().sum().sum())
print("Row retention: {:.2f}%".format(100 * processed_df.shape[0] / original_df.shape[0]))

processed_df shape: (9864, 69)
Missing values in processed_df: 0
Row retention: 100.00%


In [20]:
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

val_accuracy = model.score(X_val, y_val)
print("Validation accuracy:", val_accuracy)

Validation accuracy: 0.8667004561581348


In [22]:
predictions = model.predict(X_test_encoded)
print("Predictions shape:", predictions.shape)
print("Sample predictions:", predictions[:10])

Predictions shape: (2466,)
Sample predictions: [False False  True False False False False False False False]


In [23]:
original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()
original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape
row_retained_percent = (processed_rows / original_rows) * 100

final_model = model
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]
model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({
    "id": [
        "original_missing", "processed_missing", "original_rows",
        "processed_rows", "original_columns", "processed_columns",
        "row_retained_percent", "model_name"
    ],
    "value": [
        original_missing, processed_missing, original_rows,
        processed_rows, original_columns, processed_columns,
        round(row_retained_percent, 2), model_name
    ]
})

# Convert predictions to match sample_submission's exact casing (TRUE / FALSE, all caps)
pred_str = np.asarray(predictions).astype(str)
pred_str = np.where(pred_str == "True", "TRUE", pred_str)
pred_str = np.where(pred_str == "False", "FALSE", pred_str)

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": pred_str
})

# --- Verify conversion before saving ---
print("\n--- Converted predictions (sample) ---")
print(prediction_output.head(10))

submission = pd.concat([checkpoints, prediction_output], ignore_index=True)
submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv created successfully.")
print(checkpoints)


--- Converted predictions (sample) ---
       id  value
0  106094  FALSE
1  111845  FALSE
2  106794   TRUE
3  103444  FALSE
4  106833  FALSE
5  102684  FALSE
6  110590  FALSE
7  101688  FALSE
8  108143  FALSE
9  100700  FALSE

submission.csv created successfully.
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  69
6  row_retained_percent               100.0
7            model_name  LogisticRegression
